<a href="https://colab.research.google.com/github/MachineLearnia/Python-Machine-Learning/blob/master/28%20-%20Pr%C3%A9traitement%20de%20donn%C3%A9es%20(Corrig%C3%A9).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%javascript
IPython.OutputArea.auto_scroll_threshold = 9999


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
url = 'https://raw.githubusercontent.com/MachineLearnia/Python-Machine-Learning/master/Dataset/dataset.csv'
data = pd.read_csv(url, index_col=0, encoding = "ISO-8859-1")
data.head()


# PRE-PROCESSING

In [ ]:
df = data.copy()
df.head()


## Création des sous-ensembles (suite au EDA)

In [ ]:
missing_rate = df.isna().sum()/df.shape[0]


In [ ]:
blood_columns = list(df.columns[(missing_rate < 0.9) & (missing_rate >0.88)])
viral_columns = list(df.columns[(missing_rate < 0.80) & (missing_rate > 0.75)])


In [ ]:
key_columns = ['Patient age quantile', 'SARS-Cov-2 exam result']


In [ ]:
df = df[key_columns + blood_columns + viral_columns]
df.head()


## TrainTest - Nettoyage - Encodage

In [ ]:
from sklearn.model_selection import train_test_split


In [ ]:
trainset, testset = train_test_split(df, test_size=0.2, random_state=0)


In [ ]:
trainset['SARS-Cov-2 exam result'].value_counts()


In [ ]:
testset['SARS-Cov-2 exam result'].value_counts()


In [ ]:
def encodage(df):
    code = {'negative':0,
            'positive':1,
            'not_detected':0,
            'detected':1}
    
    for col in df.select_dtypes('object').columns:
        df.loc[:,col] = df[col].map(code)
        
    return df


In [ ]:
def feature_engineering(df):
    df['est malade'] = df[viral_columns].sum(axis=1) >= 1
    df = df.drop(viral_columns, axis=1)
    return df


In [ ]:
def imputation(df):
    #df['is na'] = (df['Parainfluenza 3'].isna()) | (df['Leukocytes'].isna())
    #df = df.fillna(-999)
    df = df.dropna(axis=0)
    return  df


In [ ]:
def preprocessing(df):
    
    df = encodage(df)
    df = feature_engineering(df)
    df = imputation(df)
    
    X = df.drop('SARS-Cov-2 exam result', axis=1)
    y = df['SARS-Cov-2 exam result']
    
    print(y.value_counts())
    
    return X, y


In [ ]:
X_train, y_train = preprocessing(trainset)


In [ ]:
X_test, y_test = preprocessing(testset)


## Modellisation

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import PolynomialFeatures
from sklearn.decomposition import PCA


In [ ]:
model_1 = RandomForestClassifier(random_state=0)


In [ ]:
model_2 = make_pipeline(PolynomialFeatures(2), SelectKBest(f_classif, k=10),
                      RandomForestClassifier(random_state=0))


## Procédure d'évaluation

In [ ]:
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from sklearn.model_selection import learning_curve


In [ ]:
def evaluation(model):
    
    model.fit(X_train, y_train)
    ypred = model.predict(X_test)
    
    print(confusion_matrix(y_test, ypred))
    print(classification_report(y_test, ypred))
    
    N, train_score, val_score = learning_curve(model, X_train, y_train,
                                              cv=4, scoring='f1',
                                               train_sizes=np.linspace(0.1, 1, 10))
    
    
    plt.figure(figsize=(12, 8))
    plt.plot(N, train_score.mean(axis=1), label='train score')
    plt.plot(N, val_score.mean(axis=1), label='validation score')
    plt.legend()
    
    

In [ ]:
print(y_train.unique())
print(y_train.dtype)


In [ ]:
y_train = y_train.astype(int)
y_test = y_test.astype(int)


In [ ]:
evaluation(model_1)


In [ ]:
pd.DataFrame(model_1.feature_importances_, index=X_train.columns).plot.bar(figsize=(12, 8))
